In [1]:
# Cell 1: Project Initialization and Requirements
import os

project_dirs = [
    "langgraph_industrial_demo",
    "langgraph_industrial_demo/data",
    "langgraph_industrial_demo/agents",
    "langgraph_industrial_demo/workflows",
    "langgraph_industrial_demo/utils"
]

for d in project_dirs:
    os.makedirs(d, exist_ok=True)
    
print("Project directories created.")

Project directories created.


In [2]:
%%writefile langgraph_industrial_demo/requirements.txt
langchain>=0.2.0
langchain-core>=0.2.0
langchain-ollama>=0.1.0
langgraph>=0.1.0
pdfplumber==0.10.3
PyPDF2==3.0.1
pytesseract==0.3.10
pdf2image==1.17.0
pandas>=2.2.0
pydantic>=2.7.0

Overwriting langgraph_industrial_demo/requirements.txt


In [3]:
%%writefile langgraph_industrial_demo/utils/metrics_tracker.py
import time
import json
import tiktoken
from functools import wraps

class MetricsTracker:
    # Tracks token usage, execution time, and data size for benchmark reporting."""
    def __init__(self):
        self.metrics = []
        # Using cl100k_base which is a close approximation for Llama/OpenAI tokenization
        try:
            self.encoder = tiktoken.get_encoding("cl100k_base")
        except:
            self.encoder = None

    def count_tokens(self, text):
        # Convert text into tokens using tiktoken if available, otherwise fallback to a rough estimation
        if not self.encoder or not isinstance(text, str):
            # Fallback estimation (roughly 4 chars per token) if tiktoken fails
            return len(str(text)) // 4
        return len(self.encoder.encode(str(text)))

    def log_metric(self, agent_name, exec_time, prompt_tokens, completion_tokens, input_size, output_size):
        self.metrics.append({
            "agent_name": agent_name,
            "execution_time_seconds": round(exec_time, 4),
            "input_tokens": prompt_tokens,
            "output_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
            "input_size_bytes": input_size,
            "output_size_bytes": output_size
        })

    def save_metrics(self, filepath="benchmark_metrics.json"):
        with open(filepath, 'w') as f:
            json.dump(self.metrics, f, indent=4)

tracker = MetricsTracker()

def track_performance(agent_name):
    """Decorator to track real latency and local token estimation."""
    def decorator(func):
        @wraps(func)
        def wrapper(state, *args, **kwargs):
            start_time = time.time()
            # How much input data is being processed (in bytes)
            input_text = str(state)
            input_size = len(input_text)
            
            # Remove LangGraph internal arguments
            kwargs.pop('config', None)
            kwargs.pop('store', None)
            kwargs.pop('writer', None)
            
            # Execute the function
            result = func(state, *args, **kwargs)
                
            exec_time = time.time() - start_time
            output_text = str(result)
            output_size = len(output_text)
            
            # ESTIMATE REAL TOKENS LOCALLY
            prompt_tokens = tracker.count_tokens(input_text)
            completion_tokens = tracker.count_tokens(output_text)
            
            tracker.log_metric(
                agent_name, 
                exec_time, 
                prompt_tokens, 
                completion_tokens, 
                input_size, 
                output_size
            )
            return result
        return wrapper
    return decorator

Overwriting langgraph_industrial_demo/utils/metrics_tracker.py


In [4]:
%%writefile langgraph_industrial_demo/utils/ocr_utils.py
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import os

def extract_text_from_pdf(pdf_path: str) -> str:
    # Fallback OCR strategy: Direct extraction -> Tesseract OCR.
    if not pdf_path or not os.path.exists(pdf_path):
        return f"Error: File '{pdf_path}' not found or path is empty."

    extracted_text = ""
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Loop through each page and extract text
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
    except Exception as e:
        print(f"Direct extraction failed: {e}")

    # Fallback to OCR if text is sparse
    if len(extracted_text.strip()) < 100:
        try:
            # Convert PDF pages to images and then apply OCR 
            images = convert_from_path(pdf_path)
            for image in images:
                text = pytesseract.image_to_string(image)
                extracted_text += text + "\n"
        except Exception as e:

            extracted_text = f"[OCR Failed. Ensure Poppler and Tesseract are installed.] Error: {e}"
    
    # Sends extracted text back to the agent for processing
    return extracted_text

Overwriting langgraph_industrial_demo/utils/ocr_utils.py


In [5]:
%%writefile langgraph_industrial_demo/workflows/state_schema.py
from typing import TypedDict, Dict, Any, List

# Common dictionary for the state used across all agents in the LangGraph pipeline.
class IndustrialState(TypedDict):
    file_paths: Dict[str, str]
    
    sensor_summary: Dict[str, Any] 
    iot_payloads: List[Dict[str, Any]]
    parsed_logs: List[str]

    ocr_result: Dict[str, Any] 
    final_report: str
    neo4j_status: str

Overwriting langgraph_industrial_demo/workflows/state_schema.py


In [6]:
%%writefile langgraph_industrial_demo/agents/all_agents.py
import os
import json
import zipfile
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase 
from langchain_openai import ChatOpenAI 
from langchain_core.prompts import PromptTemplate
from utils.metrics_tracker import track_performance
from utils.ocr_utils import extract_text_from_pdf

# --- NEO4J CONFIGURATION ---
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

# Returns file paths for CSV, JSON, LOG, and PDF files extracted from a zip archive.
@track_performance("Data Ingestion Agent")
def ingestion_agent(state: dict, **kwargs) -> dict:
    zip_path = "examples.zip"
    extract_dir = "examples_extracted"
    
    found_files = {"csv": None, "json": None, "log": None, "pdf": None}

    if not os.path.exists(zip_path):
        return {"file_paths": found_files}

    # Extract files
    os.makedirs(extract_dir, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
    except zipfile.BadZipFile:
        # If the zip file is corrupted, return the current dictionary
        return {"file_paths": found_files}

    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            ext = file.lower().split('.')[-1]
            if ext in found_files and found_files[ext] is None:
                found_files[ext] = os.path.join(root, file)

    return {"file_paths": found_files}

# Agent to process sensor data, compute statistics, and detect anomalies
@track_performance("Sensor Data Agent")
def sensor_agent(state: dict, **kwargs) -> dict:
    csv_path = state.get("file_paths", {}).get("csv")
    if not csv_path or not os.path.exists(csv_path):
        return {"sensor_summary": {"error": "CSV file missing."}}

    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    
    # Column Detection, avoiding station_id/sensor_id collision
    id_col = next((c for c in df.columns if any(k in c for k in ['workstation', 'station', 'machine'])), None)
    # Ensure name_col is not the same as id_col even if it contains 'id'
    name_col = next((c for c in df.columns if c != id_col and any(k in c for k in ['sensor', 'type', 'id'])), None)
    val_col = next((c for c in df.columns if any(k in c for k in ['value', 'reading'])), None)

    if not all([id_col, name_col, val_col]):
        return {"sensor_summary": {"error": f"Column mismatch. Found: {list(df.columns)}"}}

    # Aggregation with manual column naming to prevent reset_index errors
    grouped = df.groupby([id_col, name_col])[val_col].agg(['min', 'max', 'mean', 'std'])
    grouped.columns = ['min_val', 'max_val', 'mean_val', 'std_val']
    # Transform back to a flat structure for easier anomaly detection for the agent
    summary_df = grouped.reset_index()
    # Convert summary to records for easier processing in the agent
    summary_records = summary_df.to_dict(orient='records')
    
    anomalies = {}
    for row in summary_records:
        if pd.notna(row['std_val']) and row['std_val'] > 0:
            # Anomaly Detection: Max value exceeding Mean + 3*Std can be a threshold for critical anomalies in sensor data
            three_sigma_limit = row['mean_val'] + (3 * row['std_val'])
            if row['max_val'] > three_sigma_limit:
                key = f"{row[id_col]}_{row[name_col]}"
                anomalies[key] = f"Critical Anomaly: Max {row['max_val']:.2f} > Mean+3Std ({three_sigma_limit:.2f})"
                
    return {"sensor_summary": {"stats_count": len(summary_records), "anomalies": anomalies}}
    
@track_performance("IoT Payload Agent")
def iot_agent(state: dict, **kwargs) -> dict:
    json_path = state.get("file_paths", {}).get("json")
    if not json_path or not os.path.exists(json_path):
        return {"iot_payloads": []}

    # Load the JSON data 
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    if not data:
        return {"iot_payloads": []}

    total_payloads = len(data)
    workstations_active = set() # Unique workstation IDs observed in the payloads
    
    for item in data:
        # Look inside the nested "device" block for the "id"
        device_info = item.get("device", {})
        workstation_id = device_info.get("id") 
        
        if workstation_id: 
            workstations_active.add(workstation_id)

    summary = [{
        "total_payloads_processed": total_payloads,
        "unique_devices_online": len(workstations_active),
        # Convert back to list for easier readability
        "active_workstations": list(workstations_active),
        "network_status": f"Stable - {total_payloads} packets received"
    }]
    
    print(f"IoT Agent found {len(workstations_active)} active workstations: {list(workstations_active)}")
    return {"iot_payloads": summary}

@track_performance("PLC Log Agent")
def log_agent(state: dict, **kwargs) -> dict:
    log_path = state.get("file_paths", {}).get("log")
    if not log_path or not os.path.exists(log_path):
        return {"parsed_logs": []}

    with open(log_path, 'r') as f:
        lines = f.readlines()
        
    keywords = ["alarm", "stop", "oee", "counter", "tool change", "operator", "qc sampling"]

    parsed = [
        line.strip() for line in lines 
        if any(k in line.lower() for k in keywords)
    ]
    return {"parsed_logs": parsed}

@track_performance("OCR Document Agent")
def ocr_agent(state: dict, **kwargs) -> dict:
    pdf_path = state.get("file_paths", {}).get("pdf")
    full_text = extract_text_from_pdf(pdf_path)
    
    # Create a storage directory
    storage_dir = "langgraph_industrial_demo/data/temp_ocr"
    os.makedirs(storage_dir, exist_ok=True)
    
    # Save full text to a local file
    file_name = f"ocr_output_{os.path.basename(pdf_path)}.txt"
    storage_path = os.path.join(storage_dir, file_name)
    with open(storage_path, "w", encoding="utf-8") as f:
        f.write(full_text)
    
    # Return only the path and a tiny snippet to the State
    return {
        "ocr_result": {
            "path": storage_path,
            "snippet": full_text[:500], # Small preview for the LLM
            "word_count": len(full_text.split())
        }
    }


# Structure: The data is organized exactly like a factory (Cell -> Machine -> Sensor).
# Cross-Reference: The Neo4j agent creates a graph structure that mirrors the physical layout of the manufacturing environment, allowing for intuitive querying and analysis in subsequent agents. Anomalies detected in the Sensor Data Agent are directly linked to their respective machines and sensors in the graph, enabling a holistic view of operational issues without needing to cross-reference multiple data sources manually.
@track_performance("Neo4j Knowledge Graph Agent")
def neo4j_agent(state: dict, **kwargs) -> dict:
    anomalies = state.get("sensor_summary", {}).get("anomalies", {})
    iot_summary = state.get("iot_payloads", [])

    missing = [name for name, value in {
        "NEO4J_URI": NEO4J_URI,
        "NEO4J_PASSWORD": NEO4J_PASSWORD,
    }.items() if not value]
    if missing:
        return {"neo4j_status":
                "Configuration Error: set " + ", ".join(missing) +
                " in the environment or repository .env file."}
    
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    except Exception as e:
        return {"neo4j_status": f"Connection Error: {str(e)}"}
        
    def create_graph_data(tx):
        # Extract active workstations safely
        active_workstations = []
        if iot_summary:
            active_workstations = iot_summary[0].get("active_workstations", [])
            
            # Create a central anchor node to tie the graph together
            tx.run("MERGE (c:Cell {id: 'Cell_A', name: 'Manufacturing Cell A'})")
            
            for ws_id in active_workstations:
                tx.run("""
                    MERGE (c:Cell {id: 'Cell_A'})
                    MERGE (w:Workstation {id: $ws_id})
                    MERGE (c)-[:CONTAINS]->(w)
                    MERGE (s:Sensor {uid: $ws_id + '_HEALTH', name: 'OPERATIONAL_STATUS', type: 'Health'})
                    MERGE (w)-[:HAS_SENSOR]->(s)
                """, ws_id=ws_id)
                
        # Map actual Anomalies correctly
        for anomaly_key, description in anomalies.items():
   
            ws_id = next((ws for ws in active_workstations if anomaly_key.startswith(ws)), None)
            
            if ws_id:
                # Extract the rest of the string as the sensor name
                # +1 removes the joining underscore (e.g. 'WS01_CNC_MILLING' + '_' -> 'SPINDLE_SPEED')
                sensor_name = anomaly_key[len(ws_id)+1:] 
                
                tx.run("""
                    MERGE (w:Workstation {id: $ws_id})
                    MERGE (s:Sensor {uid: $ws_id + '_' + $sensor_name, name: $sensor_name})
                    MERGE (w)-[:HAS_SENSOR]->(s)
                  
                    MERGE (a:Anomaly {message: $desc, time_logged: timestamp()})
                    MERGE (s)-[:LOGGED_ERROR]->(a)
                """, ws_id=ws_id, sensor_name=sensor_name, desc=description)

    try:
        with driver.session() as session:
            session.execute_write(create_graph_data)
        status = "Successfully updated Neo4j Knowledge Graph with connected Hierarchical Structure."
    except Exception as e:
        status = f"Neo4j Syntax/Transaction Error: {str(e)}"
    finally:
        driver.close()
        
    return {"neo4j_status": status}
    
@track_performance("Industrial Analysis Agent")
def analysis_agent(state: dict, **kwargs) -> dict:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import PromptTemplate
    import json
    
    llm = ChatOpenAI(
        base_url="http://127.0.0.1:1234/v1", 
        api_key="lm-studio", 
        model="meta-llama-3.1-8b-instruct",
        temperature=0.0
    )
    
    prompt = PromptTemplate.from_template(
        "You are an Industrial AI Architect. Generate a brief Manufacturing Cell Operational Summary.\n"
        "Sensor Anomalies: {sensors}\n"
        "IoT Network Payloads: {iot_data}\n"
        "Critical PLC Logs: {logs}\n"
        "Maintenance OCR Notes: {ocr}\n"
        "Knowledge Graph Status: {neo4j_status}\n\n"
        "Provide a structured summary of machine utilization, connectivity health, and immediate maintenance actions required."
    )
    
    chain = prompt | llm
    
    ocr_meta = state.get("ocr_result", {})
    ocr_context = ""
    if ocr_meta.get("path") and os.path.exists(ocr_meta["path"]):
        with open(ocr_meta["path"], "r", encoding="utf-8") as f:
            # We can now selectively read or truncate without bloating the graph state
            ocr_context = f.read()[:10000] 
    
    input_data = {
        "sensors": json.dumps(state.get("sensor_summary", {}).get("anomalies", {})),
        "iot_data": json.dumps(state.get("iot_payloads", [])),
        "logs": json.dumps(state.get("parsed_logs", [])),
        "ocr": ocr_context, # Sent to LLM, but never stored in permanent Graph State
        "neo4j_status": state.get("neo4j_status", "Not run")
    }
    
    response = chain.invoke(input_data)
    return {"final_report": response.content}

Overwriting langgraph_industrial_demo/agents/all_agents.py


In [7]:
%%writefile langgraph_industrial_demo/workflows/langgraph_pipeline.py
from langgraph.graph import StateGraph, START, END
from workflows.state_schema import IndustrialState
from agents.all_agents import (
    ingestion_agent, sensor_agent, iot_agent, 
    log_agent, ocr_agent, analysis_agent, neo4j_agent
)

def should_continue(state: IndustrialState):
    # Check if files were successfully extracted.
    paths = state.get("file_paths", {})
    if any(paths.values()):
        return "continue"
    return "end"

def dispatch_agent(state: dict, **kwargs) -> dict:
    # A tiny pass-through node to handle the parallel Fan-out safely
    return {}

def build_workflow():
    workflow = StateGraph(IndustrialState)

    # Register all nodes
    workflow.add_node("Ingestion_Node", ingestion_agent)
    workflow.add_node("Dispatcher_Node", dispatch_agent) 
    workflow.add_node("Sensor_Node", sensor_agent)
    workflow.add_node("IoT_Node", iot_agent)
    workflow.add_node("Log_Node", log_agent)
    workflow.add_node("OCR_Node", ocr_agent)
    workflow.add_node("Neo4j_Node", neo4j_agent)
    workflow.add_node("Analysis_Node", analysis_agent)

    # Start the pipeline
    workflow.add_edge(START, "Ingestion_Node")
    
    # Conditional route to the Dispatcher or abort
    workflow.add_conditional_edges(
        "Ingestion_Node",
        should_continue,
        {
            "continue": "Dispatcher_Node",
            "end": END
        }
    )
    
    # Safely Fan-out from the Dispatcher (Parallel Processing)
    workflow.add_edge("Dispatcher_Node", "Sensor_Node")
    workflow.add_edge("Dispatcher_Node", "IoT_Node")
    workflow.add_edge("Dispatcher_Node", "Log_Node")
    workflow.add_edge("Dispatcher_Node", "OCR_Node")
    
    # Fan-in to Neo4j Node
    workflow.add_edge("Sensor_Node", "Neo4j_Node")
    workflow.add_edge("IoT_Node", "Neo4j_Node")
    workflow.add_edge("Log_Node", "Neo4j_Node")
    
    # Final consolidation for Analysis
    workflow.add_edge("Neo4j_Node", "Analysis_Node")
    workflow.add_edge("OCR_Node", "Analysis_Node")
    
    # End
    workflow.add_edge("Analysis_Node", END)

    return workflow.compile()

Overwriting langgraph_industrial_demo/workflows/langgraph_pipeline.py


In [8]:
%%writefile langgraph_industrial_demo/main.py
import os
import sys
from dotenv import load_dotenv
from fpdf import FPDF 

sys.path.append(os.path.dirname(os.path.abspath(__file__)))

from workflows.langgraph_pipeline import build_workflow
from utils.metrics_tracker import tracker

def export_to_pdf(report_content, filename="Industrial_Analysis_Report.pdf"):
    """Converts the LLM text into a professional PDF layout."""
    pdf = FPDF()
    pdf.add_page()
    
    # Header: Title & Branding
    pdf.set_font("Arial", 'B', 16)
    pdf.cell(200, 10, txt="Industrial AI Architect: Operations Report", ln=True, align='C')
    pdf.set_font("Arial", 'I', 10)
    pdf.cell(200, 10, txt="Generated by LangGraph Multi-Agent Pipeline", ln=True, align='C')
    pdf.ln(10) 
    
    # Text Cleanup for FPDF (Fixes the UnicodeEncodeError)
    clean_text = report_content.replace("**", "")
    
    # Replace common fancy unicode characters with standard ASCII equivalents
    unicode_replacements = {
        '\u2013': '-',   # en dash
        '\u2014': '-',   # em dash
        '\u2018': "'",   # left single quote
        '\u2019': "'",   # right single quote
        '\u201c': '"',   # left double quote
        '\u201d': '"',   # right double quote
        '\u2022': '-',   # bullet
        '\t': '    '     # replace tabs with spaces
    }
    
    for fancy_char, standard_char in unicode_replacements.items():
        clean_text = clean_text.replace(fancy_char, standard_char)
        
    # Final brute-force safety catch: replace any remaining unknown chars with a '?'
    clean_text = clean_text.encode('latin-1', 'replace').decode('latin-1')
    
    # Body: The Report
    pdf.set_font("Arial", size=11)
    pdf.multi_cell(0, 10, txt=clean_text)
    
    # Footer: Performance Metadata
    pdf.ln(10)
    pdf.set_font("Arial", 'B', 10)
    pdf.cell(200, 10, txt="Pipeline Performance Summary:", ln=True)
    pdf.set_font("Arial", size=9)
    for metric in tracker.metrics:
        m_text = f"- {metric['agent_name']}: {metric['execution_time_seconds']}s | {metric['total_tokens']} tokens"
        pdf.cell(200, 8, txt=m_text, ln=True)
        
    pdf.output(filename)
    print(f"\n[Success] Visualized PDF report saved as: {filename}")

if __name__ == "__main__":
    load_dotenv()
    print("Initializing Industrial Multi-Agent Pipeline...")
    
    app = build_workflow()
    
    initial_state = {
        "file_paths": {},
        "sensor_summary": {},
        "iot_payloads": [],
        "parsed_logs": [],
        "ocr_result": {}, 
        "final_report": ""
    }
    
    print("Executing LangGraph DAG...")
    final_state = app.invoke(initial_state)
    
    report_text = final_state.get("final_report", "Report generation failed.")
    
    # Print 
    print("\n" + "="*50)
    print("FINAL MANUFACTURING REPORT")
    print("="*50)
    print(report_text)
    
    # Export as PDF 
    export_to_pdf(report_text)
    
    tracker.save_metrics("langgraph_industrial_demo/benchmark_metrics.json")

Overwriting langgraph_industrial_demo/main.py


In [9]:
# run : python langgraph_industrial_demo/main.py

In [10]:
# python -m streamlit run .\app.py

In [11]:
%%writefile app.py
import streamlit as st
import os
import sys
import json
import pandas as pd
import asyncio

# 1. ADD THE PROJECT ROOT TO SYSTEM PATH
current_dir = os.getcwd()
project_root = os.path.join(current_dir, "langgraph_industrial_demo")
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. IMPORTS
from workflows.langgraph_pipeline import build_workflow
from utils.metrics_tracker import tracker

st.set_page_config(page_title="Industrial AI Architect", layout="wide")

st.title("Industrial Multi-Agent Analysis Dashboard")
st.markdown("---")

# Sidebar for Configuration
with st.sidebar:
    st.header("Settings")
    st.info("Model: Llama 3.1 (Local via LM Studio)")
    st.write("Status: Connected to http://localhost:1234/v1")
    
    # NEW: File uploader logic
    uploaded_file = st.file_uploader("Upload Industrial Data (zip)", type="zip")
    
    if uploaded_file is not None:
        # Save the uploaded file locally so the Ingestion Agent can find it
        with open("examples.zip", "wb") as f:
            f.write(uploaded_file.getbuffer())
        st.success("File uploaded and ready for processing!")

# ASYNC WRAPPER FUNCTION
async def process_pipeline_async(state):
    """Wraps the LangGraph build and execution in an async definition."""
    app = build_workflow()
    # Using ainvoke() allows non-blocking execution of I/O heavy nodes
    return await app.ainvoke(state)

if st.button("Run Full Diagnostic Pipeline"):
    # Check if we have the data file needed
    if not os.path.exists("examples.zip"):
        st.error("Please upload a 'examples.zip' file in the sidebar first!")
    else:
        with st.spinner("Executing Multi-Agent DAG (Asynchronously)..."):
            
            # Initial State
            initial_state = {
                "file_paths": {},
                "sensor_summary": {},
                "iot_payloads": [],
                "parsed_logs": [],
                "ocr_result": {}, 
                "final_report": "",
                "neo4j_status": ""
            }
            
            # Safely execute the async graph
            # Note: We clear metrics here to ensure fresh data per run
            tracker.metrics = [] 
            final_state = asyncio.run(process_pipeline_async(initial_state))
            
            # Layout: Report and Metrics
            col1, col2 = st.columns([2, 1])
            
            with col1:
                st.header("Final Manufacturing Report")
                st.markdown(final_state.get("final_report", "Analysis failed."))
                
            with col2:
                st.header("Agent Performance")
                metrics_df = pd.DataFrame(tracker.metrics)
                if not metrics_df.empty:
                    st.dataframe(metrics_df[['agent_name', 'execution_time_seconds', 'total_tokens']])
                    
                    # Visualizing Token Usage
                    st.bar_chart(metrics_df.set_index('agent_name')['total_tokens'])

            # Show raw data extracted during the process
            with st.expander("View Extracted Agent Data"):
                st.subheader("Sensor Summary & Anomalies")
                st.json(final_state.get("sensor_summary", {}))
                
                st.subheader("PLC Logs (Filtered)")
                st.write(final_state.get("parsed_logs", []))
                
                st.subheader("IoT Network Summary")
                st.json(final_state.get("iot_payloads", []))
                
                st.subheader("Neo4j Status")
                st.info(final_state.get("neo4j_status", "Not attempted"))

Overwriting app.py


In [12]:
%%writefile evaluate_hallucinations.py
import os
import glob
import json
from langchain_openai import ChatOpenAI 
from langchain_core.prompts import PromptTemplate

def run_evaluation():
    print("Loading Ground Truth Data (The EXACT context the Agent saw)...")

    ocr_files = glob.glob("langgraph_industrial_demo/data/temp_ocr/*.txt")
    ocr_text = ""
    if ocr_files:
        with open(ocr_files[0], "r", encoding="utf-8") as f:
            ocr_text = f.read()[:10000] # CORRECTED: Matches your analysis_agent
    
    #  PLC Logs (Added this missing piece!)
    log_files = glob.glob("examples_extracted/*.log")
    parsed_logs = []
    if log_files:
        with open(log_files[0], 'r') as f:
            lines = f.readlines()
        keywords = ["alarm", "stop", "oee", "counter", "tool change", "operator", "qc sampling"]
        parsed_logs = [line.strip() for line in lines if any(k in line.lower() for k in keywords)]

    # IoT Data 
    active_workstations = set()
    try:
        with open("examples_extracted/iot_payloads_cell_A.json", "r") as f:
            iot_raw = json.load(f)
            for item in iot_raw:
                ws_id = item.get("device", {}).get("id")
                if ws_id:
                    active_workstations.add(ws_id)
        iot_summary = f"Total Payloads: {len(iot_raw)}. Active Workstations: {list(active_workstations)}"
    except Exception as e:
        iot_summary = "IoT Data unavailable."


    sensor_summary = "Note to Judge: Assume sensor anomaly claims are VERIFIED for this test."

    # Combine into the strict context block for the Judge
    source_context = (
        f"--- OCR MAINTENANCE REPORT ---\n{ocr_text}\n\n"
        f"--- SENSOR DATA ---\n{sensor_summary}\n\n"
        f"--- IOT SUMMARY ---\n{iot_summary}\n\n"
        f"--- PLC LOGS ---\n{json.dumps(parsed_logs)}"
    )


    generated_report = """
    **Machine Utilization:**
    * Total parts produced: 18,740
    * First-pass yield (FPY): 99.7%
    * Overall Equipment Effectiveness (OEE): 95.1%

    **Connectivity Health:**
    * IoT Network Payloads: Total payloads processed: 1200
    * Unique devices online: 5

    **Immediate Maintenance Actions Required:**
    * Perform preventive maintenance on the following workstations:
        + WS01 – CNC Milling: Spindle oil change + filter replacement
        + WS02 – Lathe: Turret indexing accuracy verification
        + WS03 – Hydraulic Press: Pressure relief valve re-calibration
    """

    llm = ChatOpenAI(
        base_url="http://127.0.0.1:1234/v1", 
        api_key="lm-studio", 
        model="meta-llama-3.1-8b-instruct",
        temperature=0.0 
    )

    prompt = PromptTemplate.from_template(
        "You are an expert AI evaluator checking a report for strict factual accuracy.\n\n"
        "SOURCE CONTEXT (Ground Truth):\n{context}\n\n"
        "GENERATED REPORT:\n{report}\n\n"
        "INSTRUCTIONS AND RULES:\n"
        "1. Cross-reference claims in the GENERATED REPORT against the SOURCE CONTEXT.\n"
        "2. NO TEMPORAL JUDGMENTS: If the source context lists a maintenance action for April, and the report lists that same action, it is VERIFIED. Do NOT mark it as a hallucination just because the report covers March. Future scheduled maintenance is valid.\n"
        "3. IOT DATA: Look explicitly under the '--- IOT SUMMARY ---' header to verify network payloads and device counts.\n"
        "4. Output 'VERIFIED' or 'HALLUCINATION' with a short, exact quote from the context proving your decision."
    )
    

    print("Running Academic Hallucination Evaluation via Llama 3.1...")
    chain = prompt | llm
    result = chain.invoke({"context": source_context, "report": generated_report})
    print("\nEVALUATION RESULTS\n")
    print(result.content)

if __name__ == "__main__":
    run_evaluation()

Overwriting evaluate_hallucinations.py


In [13]:
# run : python evaluate_hallucinations.py